In [1]:
%%capture
!pip install -U transformers peft accelerate safetensors huggingface_hub
!pip install -U "torchao>=0.16.0"

## merge the model

In [2]:
import os
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel


BASE_MODEL = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

ADAPTER_PATH = (
    "/kaggle/input/notebooks/atomstack001/"
    "unsloth-deepseek-r1-distill-llama-fine-tune/"
    "llama_8b_adapter/final_adapter"
)

MERGED_PATH = (
    "/kaggle/working/"
    "deepseek-r1-distill-llama-8b-merged"
)


print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
)

print("Loading FP16 base model on CPU...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map={"": "cpu"},
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(
    model,
    ADAPTER_PATH,
)

print("Merging adapter...")
model = model.merge_and_unload(
    safe_merge=True
)

print(f"Saving merged model to {MERGED_PATH}...")
model.save_pretrained(
    MERGED_PATH,
    safe_serialization=True,
    max_shard_size="5GB",
)
tokenizer.save_pretrained(MERGED_PATH)

print("Merge complete.")
print("Saved files:", os.listdir(MERGED_PATH))

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading tokenizer...


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Loading LoRA adapter...
Merging adapter...
Saving merged model to /kaggle/working/deepseek-r1-distill-llama-8b-merged...


Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]

Merge complete.
Saved files: ['config.json', 'tokenizer_config.json', 'tokenizer.json', 'model-00004-of-00004.safetensors', 'model.safetensors.index.json', 'chat_template.jinja', 'model-00002-of-00004.safetensors', 'model-00001-of-00004.safetensors', 'generation_config.json', 'model-00003-of-00004.safetensors']
